## Epoki The Daily Forecaster - Review Notebook

This notebook loads the disbursement pipeline, runs the three candidate models (Prophet, SARIMAX, LightGBM), compares their expanding-window backtest performance, and lets you inspect forecasts and segment scores interactively.

Run each cell in order. You can swap the CSV path or the column names in the first code cell.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from daily_forecaster import (
    load_data,
    load_custom_holidays,
    build_features,
    build_tanzania_holiday_table,
    run_backtests,
    summarize_backtest,
    compute_ensemble_weights,
    generate_final_forecast,
    aggregate_monthly,
    FeatureConfig,
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# -------------------- User inputs --------------------
CSV_PATH = 'data/sample_data.csv'          # change to your file
DATE_COL = 'date'
VALUE_COL = 'disbursement_amount'
# Set to True to include the strike/abnormal event features, False to ignore them
INCLUDE_STRIKE = True

STRIKE_START = '2025-10-29' if INCLUDE_STRIKE else None
STRIKE_END = '2025-11-02' if INCLUDE_STRIKE else None

# Set to a date like '2026-09-01' if you want the forecast to start
# on a specific planning date instead of the day after the last history date.
FORECAST_START_DATE = None   # set to None to start day after last history date

df = load_data(CSV_PATH, DATE_COL, VALUE_COL)
df.tail()

2026-09-24 22:19:43,830 | INFO | Loaded 1339 daily observations from 2023-01-01 to 2026-08-31.


,ds,y,is_imputed
1334,2026-08-27,137.58,0
1335,2026-08-28,155.21,0
1336,2026-08-29,67.09,0
1337,2026-08-30,68.98,0
1338,2026-08-31,155.03,0


## 1. Inspect the raw history

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(12, 6))

axes[0].plot(df['ds'], df['y'], label='daily disbursement')
axes[0].set_title('Daily disbursement history')
axes[0].set_ylabel('Amount')
axes[0].legend()

weekly = df.set_index('ds').resample('W')['y'].sum()
axes[1].plot(weekly.index, weekly.values, color='C1', label='weekly total')
axes[1].set_title('Weekly disbursement totals')
axes[1].set_ylabel('Amount')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Build features and holiday calendar

In [3]:
cfg = FeatureConfig(strike_start=STRIKE_START, strike_end=STRIKE_END)
years = sorted(set(df['ds'].dt.year) | {df['ds'].max().year + 1, df['ds'].max().year + 2})

# -------------------- Optional custom holidays --------------------
# If you have a holiday.csv with columns 'date' and 'holiday', set the path below.
# Otherwise leave HOLIDAY_CSV_PATH as None to use only the built-in Tanzania calendar.
HOLIDAY_CSV_PATH = 'data/holiday.csv'   # change to your file or set to None

if HOLIDAY_CSV_PATH:
    custom_holidays = load_custom_holidays(HOLIDAY_CSV_PATH)
    print(f'Loaded {len(custom_holidays)} custom holiday(s).')
else:
    custom_holidays = None

holiday_table = build_tanzania_holiday_table(years, custom_holidays)

feat_df = build_features(df, cfg, holiday_table)
print('Feature matrix shape:', feat_df.shape)
print('Columns:', feat_df.columns.tolist()[:20], '...')
feat_df.head()

Loaded 67 custom holiday(s).
Feature matrix shape: (1339, 54)
Columns: ['ds', 'y', 'is_imputed', 'day_of_week', 'day_name', 'is_sunday', 'is_weekend', 'day_of_month', 'week_of_year', 'month', 'quarter', 'year', 'is_month_end', 'is_month_start', 'is_salary_period', 'is_holiday', 'is_day_before_holiday', 'is_day_after_holiday', 'days_to_holiday', 'days_from_holiday'] ...


,ds,y,is_imputed,day_of_week,day_name,is_sunday,is_weekend,day_of_month,week_of_year,month,...,rolling_mean_90,rolling_std_90,rolling_mean_365,rolling_std_365,holtype_christmas,holtype_custom,holtype_easter,holtype_eid,holtype_none,holtype_public_holiday
0,2023-01-01,49.56,0,6,Sunday,1,1,1,52,1,...,NaN,NaN,NaN,NaN,False,True,False,False,False,False
1,2023-01-02,102.71,0,0,Monday,0,0,2,1,1,...,NaN,NaN,NaN,NaN,False,False,False,False,True,False
2,2023-01-03,107.47,0,1,Tuesday,0,0,3,1,1,...,NaN,NaN,NaN,NaN,False,False,False,False,True,False
3,2023-01-04,92.03,0,2,Wednesday,0,0,4,1,1,...,NaN,NaN,NaN,NaN,False,False,False,False,True,False
4,2023-01-05,81.52,0,3,Thursday,0,0,5,1,1,...,NaN,NaN,NaN,NaN,False,False,False,False,True,False


## 3. Run expanding-window backtests

In [ ]:
bt_results = run_backtests(
    feat_df,
    holiday_table,
    cfg,
    horizon_days=90,
    step_months=3,
    min_train_months=12,
)

summary = summarize_backtest(bt_results)
summary

## 4. Model comparison plot

In [ ]:
metrics_to_plot = ['MAE', 'RMSE', 'WAPE', 'sMAPE']
summary_melted = summary.melt(id_vars='Model', value_vars=metrics_to_plot, var_name='metric', value_name='value')

g = sns.catplot(
    data=summary_melted, x='Model', y='value', col='metric',
    kind='bar', sharey=False, height=4, aspect=1.1
)
g.set_titles('{col_name}')
g.fig.suptitle('Backtest model comparison', y=1.02, fontsize=14)
plt.show()

## 5. Ensemble weights and champion

In [ ]:
weights = compute_ensemble_weights(summary, metric='WAPE')
champion = summary.sort_values('WAPE').iloc[0]['Model']

print('Champion model by WAPE:', champion)
print('Inverse-error ensemble weights:', weights)

if weights:
    pd.Series(weights).plot(kind='barh', title='Ensemble weights (inverse WAPE)')
    plt.show()

## 6. Segment scores for the champion

In [ ]:
segment_df = bt_results[champion]['fold_segments'][-1] if bt_results[champion]['fold_segments'] else None
if segment_df is not None and not segment_df.empty:
    display(segment_df)
    segment_df.set_index('segment')[['MAE', 'WAPE']].plot(kind='barh', title=f'Segment scores - {champion}')
    plt.show()
else:
    print('No segment scores available.')

## 7. Generate final forecasts (12m & 24m)

In [ ]:
horizons = {'12m': 365, '24m': 730}
forecasts = {}
monthly_forecasts = {}

for label, horizon_days in horizons.items():
    print(f'Generating {label} forecast with {champion}...')
    fc = generate_final_forecast(
        df, cfg, horizon_days, champion, weights, custom_holidays, FORECAST_START_DATE
    )
    forecasts[label] = fc.sort_values('ds').reset_index(drop=True)
    monthly_forecasts[label] = aggregate_monthly(forecasts[label])

forecasts['12m'].tail()

## 8. Plot daily forecasts with prediction bands

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, (label, fc) in zip(axes, forecasts.items()):
    ax.plot(fc['ds'], fc['yhat'], label='P50 forecast', color='C0')
    ax.fill_between(fc['ds'], fc['yhat_lower'], fc['yhat_upper'], color='C0', alpha=0.2, label='P10-P90 band')
    ax.set_title(f'{label} daily forecast')
    ax.set_xlabel('Date')
    ax.set_ylabel('Disbursement')
    ax.legend()

plt.tight_layout()
plt.show()

## 9. Plot monthly aggregates

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (label, monthly) in zip(axes, monthly_forecasts.items()):
    x = monthly['month'].astype(str)
    ax.bar(x, monthly['forecast_disbursement'], color='C0', alpha=0.7, label='P50')
    ax.bar(x, monthly['lower'], color='C1', alpha=0.3, label='P10')
    ax.bar(x, monthly['upper'], color='C2', alpha=0.3, label='P90')
    ax.set_title(f'{label} monthly forecast')
    ax.set_xlabel('Month')
    ax.set_ylabel('Total disbursement')
    ax.tick_params(axis='x', rotation=45)
    ax.legend()

plt.tight_layout()
plt.show()

## 10. Save artifacts

In [ ]:
from pathlib import Path

outdir = Path('notebook_output')
outdir.mkdir(exist_ok=True)

summary.to_csv(outdir / 'model_comparison.csv', index=False)
if segment_df is not None:
    segment_df.to_csv(outdir / 'segment_scores_last_fold.csv', index=False)
for label, fc in forecasts.items():
    fc.to_csv(outdir / f'daily_forecast_{label}.csv', index=False)
    monthly_forecasts[label].to_csv(outdir / f'monthly_forecast_{label}.csv', index=False)

print('Artifacts saved to', outdir.resolve())